# CEH GEAR Monthly Rainfall: NetCDF via R

**Launch this notebook:**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/YOUR_REPO/blob/main/gear_netcdf_r.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/YOUR_ORG/YOUR_REPO/HEAD?labpath=gear_netcdf_r.ipynb)
[![Open in DataLabs](https://img.shields.io/badge/Open%20in-NERC%20DataLabs-blue?logo=jupyter)](https://datalabs.nerc.ac.uk)

> **Google Colab:** Switch to an R runtime — Runtime → Change runtime type → R
> Replace `YOUR_ORG/YOUR_REPO` with your GitHub path once published.

---

## What this notebook does

This notebook explores the **CEH GEAR monthly rainfall** dataset — 1 km gridded estimates of monthly rainfall for Great Britain — using R.

**We will:**
1. Download and open a NetCDF file
2. Inspect its metadata (dimensions, variables, attributes)
3. Extract and plot a time series at a single location
4. Plot a map of rainfall for a single month

**Data:** 1 km monthly rainfall, Great Britain, 1890–2019  
**Source:** CEH-GEAR via EIDC  
**CRS:** British National Grid (EPSG:27700) — coordinates are easting/northing in metres

**Library used:** `ncdf4` — the standard R NetCDF package  
Plotting uses base R graphics (no extra packages needed).

| Runtime | Notes |
|---------|-------|
| Google Colab (R runtime) | ✅ Switch runtime to R |
| Binder | ✅ Full support (add `binder/install.R`) |
| NERC DataLabs | ✅ Full support |
| Local Jupyter (IRkernel) | ✅ Full support |
| RStudio / Quarto | ✅ Copy code into `.R` or `.qmd` |
| JupyterLite (webR) | ⚠️ `ncdf4` may not compile in WASM — use Binder |

---

## 0. Setup

Install and load `ncdf4`. This takes ~1 minute on first run.

In [ ]:
if (!requireNamespace("ncdf4", quietly = TRUE))
  install.packages("ncdf4", repos = "https://cloud.r-project.org")

library(ncdf4)
cat("ncdf4 version:", as.character(packageVersion("ncdf4")), "\n")

---
## 1. Download the NetCDF file

We download a single year (1894) of monthly GEAR data.  
Each file contains 12 monthly time steps for the full 1 km GB grid.

> **Note:** The file is ~60 MB. The download takes ~1 minute.

In [ ]:
nc_url  <- paste0(
  "https://catalogue.ceh.ac.uk/datastore/eidchub/",
  "dbf13dd5-90cd-457a-a986-f2f9dd97e93c/GB/monthly/CEH_GEAR_monthly_GB_1894.nc"
)
nc_file <- "CEH_GEAR_monthly_GB_1894.nc"

if (!file.exists(nc_file)) {
  cat("Downloading file (~60 MB)...\n")
  download.file(nc_url, nc_file, mode = "wb", quiet = FALSE)
} else {
  cat("File already downloaded.\n")
}

cat(sprintf("File size: %.1f MB\n", file.size(nc_file) / 1e6))

---
## 2. Open the file and inspect metadata

In [ ]:
nc <- nc_open(nc_file)

# Print the full file summary — similar to ncdump -h
cat("=== FILE SUMMARY ===\n")
cat("Format   :", nc$format, "\n")
cat("Variables:", nc$nvars, "\n")
cat("Dims     :", nc$ndims, "\n")

### 2a. Dimensions

In [ ]:
cat("=== DIMENSIONS ===\n")
for (dim in nc$dim) {
  cat(sprintf("  %-10s : %d values\n", dim$name, dim$len))
}

### 2b. Variables

In [ ]:
cat("=== VARIABLES ===\n")
for (v in nc$var) {
  dim_str <- paste(sapply(v$dim, function(d) d$name), collapse = " × ")
  units   <- ncatt_get(nc, v$name, "units")$value
  lname   <- ncatt_get(nc, v$name, "long_name")$value
  cat(sprintf("  %-30s [%s]\n", v$name, dim_str))
  if (!is.logical(units)) cat(sprintf("    units     : %s\n", units))
  if (!is.logical(lname)) cat(sprintf("    long_name : %s\n", lname))
}

### 2c. Global attributes

In [ ]:
cat("=== GLOBAL ATTRIBUTES ===\n")
gatts <- ncatt_get(nc, 0)
for (nm in names(gatts)) {
  val <- as.character(gatts[[nm]])
  if (nchar(val) > 90) val <- paste0(substr(val, 1, 87), "...")
  cat(sprintf("  %-30s : %s\n", nm, val))
}

---
## 3. Read coordinates

The GEAR NetCDF uses **British National Grid** — `x` is easting and `y` is northing in metres.

In [ ]:
x    <- ncvar_get(nc, "x")     # easting, metres (BNG)
y    <- ncvar_get(nc, "y")     # northing, metres (BNG)
time <- ncvar_get(nc, "time")  # time values

# Decode time axis
time_units <- ncatt_get(nc, "time", "units")$value
cat("Time units:", time_units, "\n")

# Parse "days since YYYY-MM-DD" format
origin <- as.Date(sub(".*since\\s+", "", time_units))
dates  <- origin + as.integer(time)

cat(sprintf("x (easting) : %d values, %.0f m to %.0f m  (%.0f m spacing)\n",
  length(x), min(x), max(x), x[2] - x[1]))
cat(sprintf("y (northing): %d values, %.0f m to %.0f m\n",
  length(y), min(y), max(y)))
cat(sprintf("Time        : %d steps, %s to %s\n",
  length(dates), min(dates), max(dates)))

---
## 4. Time series at a single location

We find the nearest grid cell to a target BNG coordinate, then read all 12 monthly values for that cell.

In [ ]:
# Target location in British National Grid (easting, northing in metres)
# Edinburgh ~ (325000, 673000)
# Convert lon/lat to BNG at: https://gridreferencefinder.com
target_x <- 325000
target_y <- 673000

# Find the nearest grid cell indices
i_x <- which.min(abs(x - target_x))
i_y <- which.min(abs(y - target_y))

cat(sprintf("Target  : x=%d, y=%d\n", target_x, target_y))
cat(sprintf("Nearest : x=%.0f (index %d), y=%.0f (index %d)\n",
  x[i_x], i_x, y[i_y], i_y))

# Read all time steps for this single grid cell
# start = [x_index, y_index, time_index_1], count = [1, 1, all_times]
ts_vals <- ncvar_get(
  nc, "rainfall_amount",
  start = c(i_x, i_y, 1),
  count = c(1, 1, length(time))
)

ts_df <- data.frame(date = dates, rainfall_mm = as.numeric(ts_vals))
print(ts_df)

In [ ]:
units    <- ncatt_get(nc, "rainfall_amount", "units")$value
longname <- ncatt_get(nc, "rainfall_amount", "long_name")$value

barplot(
  ts_df$rainfall_mm,
  names.arg = format(ts_df$date, "%b"),
  col  = "#1d6fa4",
  main = paste("CEH GEAR Monthly:", longname),
  xlab = format(dates[1], "%Y"),
  ylab = paste0(longname, " (", units, ")"),
  border = NA,
  las = 1
)
mtext(
  sprintf("BNG: x=%.0f m, y=%.0f m", x[i_x], y[i_y]),
  side = 3, line = -1.5, adj = 0.02, cex = 0.8, col = "grey50"
)

---
## 5. Map at a single time step

Read the full 2-D spatial grid for the first month and plot it.

In [ ]:
# Read spatial slice for the first time step
# start = [all_x, all_y, time_step_1], count = [all, all, 1]
t_index <- 1
grid <- ncvar_get(
  nc, "rainfall_amount",
  start = c(1, 1, t_index),
  count = c(-1, -1, 1)        # -1 means read all
)

cat(sprintf("Grid dimensions: %d x (easting) × %d y (northing)\n", dim(grid)[1], dim(grid)[2]))
cat(sprintf("Value range    : %.1f to %.1f %s\n",
  min(grid, na.rm = TRUE), max(grid, na.rm = TRUE), units))

In [ ]:
# The grid from ncvar_get is [x, y] — we transpose it to [y, x] for image()
# and flip y so north is at the top (y increases downward in ncvar_get output)
grid_plot <- t(grid)[nrow(t(grid)):1, ]

# Colour palette: white → blue, similar to YlGnBu
rain_cols <- colorRampPalette(c("white", "#c6dbef", "#4393c3", "#08306b"))(100)

image(
  x = x / 1000,     # convert metres to km for cleaner axis labels
  y = rev(y) / 1000,
  z = grid_plot,
  col  = rain_cols,
  xlab = "Easting (km, BNG)",
  ylab = "Northing (km, BNG)",
  main = paste0("CEH GEAR Monthly: ", longname, "\n", format(dates[t_index], "%B %Y"))
)

# Add the selected point
points(x[i_x] / 1000, y[i_y] / 1000, pch = 3, col = "red", cex = 2, lwd = 2)

# Simple colourbar using legend
legend("bottomright",
  legend = round(seq(min(grid, na.rm=TRUE), max(grid, na.rm=TRUE), length.out = 5)),
  fill   = rain_cols[c(1, 25, 50, 75, 100)],
  title  = units, bty = "n", cex = 0.8
)

---
## 6. Annual total map

Read all 12 months and sum them to get the annual total.

In [ ]:
# Read all time steps at once: result is [x, y, time]
all_months <- ncvar_get(nc, "rainfall_amount")

# Sum over the time dimension (dim 3)
annual_total <- apply(all_months, c(1, 2), sum, na.rm = TRUE)

# Transpose and flip for plotting (same as above)
annual_plot <- t(annual_total)[nrow(t(annual_total)):1, ]

image(
  x = x / 1000,
  y = rev(y) / 1000,
  z = annual_plot,
  col  = rain_cols,
  xlab = "Easting (km, BNG)",
  ylab = "Northing (km, BNG)",
  main = paste0("CEH GEAR: Annual total rainfall ", format(dates[1], "%Y"))
)

legend("bottomright",
  legend = round(seq(min(annual_total, na.rm=TRUE), max(annual_total, na.rm=TRUE), length.out = 5)),
  fill   = rain_cols[c(1, 25, 50, 75, 100)],
  title  = units, bty = "n", cex = 0.8
)

# Always close the NetCDF connection when finished
nc_close(nc)
cat("✓ NetCDF file closed\n")

---
## Tips

| Task | Code |
|------|------|
| Read a different year | Change `1894` in the URL to any year 1890–2019 |
| Read a specific month | Change `t_index` (1 = Jan, 12 = Dec) |
| Convert BNG to lon/lat | Read the `lat` and `lon` 2-D arrays from the same file |
| Spatial mean over GB | `mean(grid, na.rm = TRUE)` |
| Check fill value | `ncatt_get(nc, "rainfall_amount", "_FillValue")` |

**Data citation:** CEH-GEAR, UKCEH. https://doi.org/10.5285/dbf13dd5-90cd-457a-a986-f2f9dd97e93c